<a href="https://colab.research.google.com/github/sahil2448/Disease-Predictor-/blob/main/Disease_Predictor_bootcamp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Day 01

In [ ]:
from google.colab import files
files.upload()

Saving heart_dataset - heart_dataset.csv.csv to heart_dataset - heart_dataset.csv (1).csv


{'heart_dataset - heart_dataset.csv (1).csv': b'age,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,sex_Female,sex_Male,cp_asymptomatic,cp_atypical angina,cp_non-anginal,cp_typical angina\r\n58,130,220,1,normal,150,FALSE,1.4,flat,0,fixed defect,0,1,0,0,0,1\r\n67,160,276,0,lv hypertrophy,108,TRUE,1.5,flat,3,normal,0,1,1,0,0,0\r\n42,120,230,0,normal,170,FALSE,1,upsloping,0,reversable defect,1,0,0,0,1,0\r\n50,130,210,0,lv hypertrophy,158,FALSE,0.8,flat,0,normal,0,1,0,0,1,0\r\n45,114,230,0,normal,165,FALSE,1.1,downsloping,0,normal,1,0,0,1,0,0'}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets download -d redwankarimsony/heart-disease-data -p /content/heart-disease --unzip

In [ ]:
import pandas as pd
df = pd.read_csv('/content/heart-disease/heart_disease_uci.csv')

In [ ]:
df.head()

In [ ]:
print(df.columns)

In [ ]:
df.isnull().sum()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df[numeric_cols].hist(figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(df[numeric_cols].corr(),annot=True,cmap='coolwarm')
plt.title('Numeric Feature Correlations')
plt.show()

### Day2 Training a Model

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if 'num' in cat_cols:
  cat_cols.remove('num')

# explain the above code:
# --> This code snippet first identifies columns with object data types and stores their names in a list called cat_cols. It then checks if the target variable 'num' is present in this list and removes it if it is, ensuring that 'num' is not treated as a categorical feature for further processing.

In [ ]:
X = df.drop('num',axis=1)
y = (df['num']>0).astype(int)

# X = df.drop('num',axis=1): This line creates a new DataFrame X by dropping the 'num' column from the original DataFrame df. axis=1 specifies that we are dropping a column. This X DataFrame will contain the features we'll use to train the model.
# y = (df['num']>0).astype(int): This line creates a new Series y which is our target variable. It takes the 'num' column from the original DataFrame df and converts it into a binary target (0 or 1). If the value in 'num' is greater than 0 (indicating the presence of heart disease), the corresponding value in y will be 1; otherwise, it will be 0. This prepares the target variable for binary classification.

In [ ]:
X = pd.get_dummies(X,columns=cat_cols)
print("Final feature columns:",X.columns)

 #This code converts the categorical features in your dataset into a numerical format that can be used by machine learning models, and then prints the names of the resulting features.

# Day-3 : Advanced model and feature engineering

In [ ]:
from sklearn.model_selection import train_test_split # used to split your dataset into training and testing sets. This is a standard practice in machine learning to evaluate the model's performance on unseen data.
from sklearn.preprocessing import StandardScaler  # This class is used for standardizing features by removing the mean and scaling to unit variance. This is often a crucial preprocessing step for many machine learning algorithms as it can help improve their performance.

In [ ]:
X_train, X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42) # This code splits your data (X and y) into training and testing sets (X_train, X_test, y_train, y_test). 80% of the data is for training, and 20% is for testing (test_size=0.2). random_state=42 ensures the split is consistent. This is done to evaluate your model on data it hasn't seen during training.

new value = (x-mean)/standard deviation

In [ ]:
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

# scalar = StandardScaler() creates a scaler object.
# X_train_scaled = scalar.fit_transform(X_train) calculates the mean and standard deviation from the training data (fit) and then applies the scaling to the training data (transform).
# X_test_scaled = scalar.transform(X_test) applies the scaling calculated from the training data to the testing data.

In [ ]:
from sklearn.linear_model import LogisticRegression # it's about classification



In [ ]:
lr_model = LogisticRegression() # Giving Admission to a new student
lr_model.fit(X_train_scaled,y_train) # training step

Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score,classification_report

In [ ]:
y_pred_lr = lr_model.predict(X_test_scaled)
print("Logistic Regression Accuracy: ",accuracy_score(y_test,y_pred_lr))
print(classification_report(y_test , y_pred_lr))

Day 4: Random Forest, and Feature Importance

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
cm = confusion_matrix(y_test,y_pred_lr)
sns.heatmap(cm,annot=True,fmt='d', cmap='Blues')
plt.title('Confusion Matrix(Logistic Regression)')
plt.show()

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train_scaled,y_train)
y_pred_lr = rf_model.predict(X_test_scaled)
print("Random Forest Accuracy: ",accuracy_score(y_test,y_pred_lr))

Feature Importance

In [ ]:
feat_imp = pd.Series(rf_model.feature_importances_,index=X.columns)
feat_imp.nlargest(10).plot(kind='barh')
plt.title('Random Forest Feature Importance')
plt.show()

#This code visualizes the top 10 most important features identified by the Random Forest model, showing their relative impact on the model's predictions.



Save the Model

In [ ]:
import joblib
joblib.dump(rf_model,'heart_rf_model.pkl')
joblib.dump(scalar,'hear_scalar.pkl')

# import joblib: Imports the joblib library, which is useful for saving and loading Python objects, especially large ones like machine learning models.
# joblib.dump(rf_model,'heart_rf_model.pkl'): This line saves the trained Random Forest model (rf_model) to a file named 'heart_rf_model.pkl'. The '.pkl' extension is a common convention for pickle files, which is the format joblib uses.
# joblib.dump(scalar,'hear_scalar.pkl'): This line saves the fitted StandardScaler object (scalar) to a file named 'hear_scalar.pkl'. It's important to save the scaler as well, because you'll need to use the same scaling transformation on any new data you want to predict on in the future.

In [ ]:
sample= X.head(1)
sample.to_csv('Heart_user_template.csv',index=False)
print("User Template saved as 'Heart_user_template.csv'")
# This code creates a sample row from your feature data (X), saves it to a CSV file named Heart_user_template.csv, and then prints a confirmation message. This template can be used to format new data for prediction with your trained model.

# Day5

In [ ]:
from google.colab import files
files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import joblib
import pandas as pd
user_df = pd.read_csv("heart_dataset - heart_dataset.csv.csv")

# Getting columns list from training dataFrame
numeric_cols =df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
bool_cols = df.select_dtypes(include='bool').columns.tolist()

# Dropping columns which are extra in user_df than required to avoid error
numeric_cols = [col for col in numeric_cols if col in user_df.columns]
cat_cols = [col for col in cat_cols if col in user_df.columns]
bool_cols = [col for col in bool_cols if col in user_df.columns]

# Fill the missing numeric column
user_df[numeric_cols]=user_df[numeric_cols].fillna(user_df[numeric_cols].mean())

for col in cat_cols:
  user_df[col]=user_df[col].fillna("Unknown")

for col in bool_cols:
  user_df[col]=user_df[col].astype(int)

# One-hot encoding cat columns---> categorical to numeric
user_df_encoded=pd.get_dummies(user_df,columns=cat_cols)

#Align Columns
user_df_encoded = user_df_encoded.reindex(columns=X.columns,fill_value=0)

#Scale data
scalar = joblib.load('hear_scalar.pkl')
user_df_scaled = scalar.transform(user_df_encoded)

#prediction
model=joblib.load('heart_rf_model.pkl')
preds = model.predict(user_df_scaled)
user_df['Heart_Disease_prediction']=preds

#Show result
print(user_df)
